In [16]:
from enum import Enum

import numpy as np
import pygame

import gymnasium as gym
from gymnasium import spaces

import importlib

from DnB import DotsAndBoxes, get_render_desc, draw_board

In [17]:
action_space = spaces.MultiDiscrete(nvec=[2,6,6], dtype=int)
print(action_space)
print(action_space.sample())

MultiDiscrete([2 6 6])
[0 3 0]


In [18]:
observation_space = spaces.Dict(
        {
            "h_edges": spaces.Box(-1, 1, shape=(6,5), dtype=int),
            "v_edges": spaces.Box(-1, 1, shape=(5,6), dtype=int),
            "box_owner": spaces.Box(-1, 1, shape=(5,5), dtype=int)
        }
    )

In [19]:
print(observation_space['h_edges'].sample())
print(observation_space['v_edges'].sample())
print(observation_space['box_owner'].sample())

[[-1  1  1  0  0]
 [ 0  0  1  0  1]
 [ 1  0  1  1  1]
 [ 1 -1 -1 -1  1]
 [ 0  1  1 -1  1]
 [ 0  0  1  1  0]]
[[ 1 -1  0  1 -1  0]
 [ 1  1  0 -1  0  0]
 [ 1  0  1  1 -1  1]
 [ 0  1  0  1 -1  1]
 [-1  0 -1 -1 -1 -1]]
[[-1  1  1  0  0]
 [ 0 -1  1 -1  0]
 [ 1 -1 -1  1  0]
 [-1  0 -1  1  0]
 [-1  1  0 -1  0]]


In [ ]:
def interpret_edges(edges):
    def map_func(e):
        if e == 0:
            return -1
        elif e == None:
            return 0
        else:
            return e

    i_edges = [[map_func(e) for e in r] for r in edges]

    return i_edges

# 여기선는 유저를 -1, 1로 구분
# DnB에서는 0, 1로 구분
def interpret_box_owner(box_owner):
    def map_func(box):
        if box == 0:
            return -1
        elif box == None:
            return 0
        else :
            return box 
        
    for r in range(len(box_owner)):
        for c, e in enumerate(box_owner[r]):
            if e == 0:
                box_owner[r][c] = -1
    i_box_owner = [[map_func(box) for box in r] for r in box_owner]
    return i_box_owner


def interpret_action(action):
    ori = 'H' if action[0] == 0 else 'V'
    r, c = map(int, action[1:])
    return (ori, r, c)

In [21]:
DnB = DotsAndBoxes(5)
DnB.claim_edge(('V', 1, 1))
DnB.claim_edge(('H', 1, 1))
DnB.claim_edge(('H', 2, 3))

print("DnB_h_Edges\n", DnB.h_edges)
print("Interpreted h_Edges\n", interpret_edges(DnB.h_edges))

# Test For Deep Copy issue
print("DnB_v_Edges\n", DnB.h_edges)
print("Interpreted v_Edges\n", interpret_edges(DnB.h_edges))

1
0
DnB_h_Edges
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, 0, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None]]
Interpreted h_Edges
 [[0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, -1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
DnB_v_Edges
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, 0, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None]]
Interpreted v_Edges
 [[0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, -1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [22]:
DnB = DotsAndBoxes(5)
action = action_space.sample()
i_action = interpret_action(action)
print(i_action)
DnB.claim_edge(i_action)
print("DnB_h_Edges\n", DnB.h_edges)


action = action_space.sample()
i_action = interpret_action(action)
print(i_action)
DnB.claim_edge(i_action)
print("DnB_h_Edges\n", DnB.h_edges)

[0 5 4]
('H', 5, 4)
0
DnB_h_Edges
 [[None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, 0]]
[0 1 1]
('H', 1, 1)
1
DnB_h_Edges
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, 0]]


In [23]:
DnB = DotsAndBoxes(5)

DnB.claim_edge(('V', 1, 1))
DnB.claim_edge(('H', 1, 1))
DnB.claim_edge(('H', 2, 1))
DnB.claim_edge(('V', 1, 2))

print("DnB_box_owner\n", DnB.box_owner)
print("Interpreted box_owner\n", interpret_box_owner(DnB.box_owner))

# Test For Deep Copy issue
print("DnB_box_owner\n", DnB.box_owner)
print("Interpreted box_owner\n", interpret_box_owner(DnB.box_owner))

1
0
DnB_box_owner
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None]]
Interpreted box_owner
 [[0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
DnB_box_owner
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None]]
Interpreted box_owner
 [[0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [24]:
action = action_space.sample()
print(action)
print(interpret_action(action))

[0 4 4]
[0 4 4]
('H', 4, 4)


In [ ]:
class DnBEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, render_mode=None, n_box = 5):

        
        self.n_box = n_box # grid size
        self.DnB = None
        self.window_size = 512 #The size of the Pygame window

        self.observation_space = spaces.Dict(
        {
            "h_edges": spaces.Box(-1, 1, shape=(6,5), dtype=int),
            "v_edges": spaces.Box(-1, 1, shape=(5,6), dtype=int),
            "box_owner": spaces.Box(-1, 1, shape=(5,5), dtype=int)
        }
    )
        
        self.h_board = np.zeros(shape=(6,5), dtype=bool)
        self.v_board = np.zeros(shape=(5,6), dtype=bool)

        self.action_space = spaces.MultiDiscrete(nvec=[2,6,6], dtype=int)


        assert render_mode is None or render_mode in self.metadata["render_modes"]
        self.render_mode = render_mode

        """
        If human-rendering is used, `self.window` will be a reference
        to the window that we draw to. `self.clock` will be a clock that is used
        to ensure that the environment is rendered at the correct framerate in
        human-mode. They will remain `None` until human-mode is used for the
        first time.
        """
        
        
        self.screen = None
        self.clock = None
        self.fonts = None


    def _get_obs(self):
        # potential Copy issue
        obs = {
            'h_edges': interpret_edges(self.DnB.h_edges),
            'v_edges': interpret_edges(self.DnB.v_edges),
            'box_owner': interpret_box_owner(self.DnB.box_owner)
        }

        return obs

    ## auxilary information. manhattan distance
    def _get_info(self):
        return {
        }
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.DnB = DotsAndBoxes(self.n_box)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info
    
    def render(self):
        if self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        if self.screen is None and self.clock is None and self.render_mode == 'human':
            pygame.init()
            pygame.display.init()
            self.screen, self.clock, self.fonts = get_render_desc(self.n_box)

        if self.render_mode == "human":
            draw_board(self.screen, self.DnB, self.fonts)
            pygame.event.pump()
            pygame.display.update()
            
            self.clock.tick(self.metadata['render_fps'])

        else:
            return np.transpose(
                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)
            )
        
    def step(self, action):
        
        i_action = interpret_action(action)
        self.DnB.claim_edge(i_action)
        
        terminated = self.DnB.is_game_over
        reward = 1 if terminated else 0
        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, False, info
    
    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

In [26]:
def get_action_sample_mask(n_box):
    # shape: (n_box+1, n_box+1)
    r = np.arange(n_box + 1)[:, None]        # (R,1)
    c = np.arange(n_box + 1)[None, :]        # (1,C)

    mask = np.zeros((2, n_box + 1, n_box + 1), dtype=bool)
    # Horizontal mask: True only when c == n_box
    mask[0] = (c == n_box)

    # Vertical mask: True only when r == n_box
    mask[1] = (r == n_box)
    # Stack so that mask[0] = H, mask[1] = V
    return mask

In [27]:
mask = get_action_sample_mask(5)
print(mask.shape)
print(mask)
print(mask[*(0, 5, 0)])

(2, 6, 6)
[[[False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]]

 [[False False False False False False]
  [False False False False False False]
  [False False False False False False]
  [False False False False False False]
  [False False False False False False]
  [ True  True  True  True  True  True]]]
False


In [ ]:
n_box = 5
env = DnBEnv(render_mode='human', n_box=n_box)
mask = get_action_sample_mask(n_box)

observation, info = env.reset()

print(f"Starting observation: {observation}")

def update_action_mask(observation):    
    for r in range(len(observation['h_edges'])):
        for c, e in enumerate(observation['h_edges'][r]):
            mask[0, r, c] |= e
    for r in range(len(observation['v_edges'])):
        for c, e in enumerate(observation['v_edges'][r]):
            mask[1, r, c] |= e
    
episode_over = False
total_reward = 0

while not episode_over:

    action = env.action_space.sample()
    while mask[*action]:
        action = env.action_space.sample()
        
    print('action:', action)
    print('Number of Claimed Edges:', np.sum(mask == False))

    observation, reward, terminated, truncated, info = env.step(action)
    
    update_action_mask(observation)
            
    total_reward += reward
    episode_over = terminated or truncated

print(f"Episode finished! Total reward: {total_reward}")
print(f'Action spasce: {env.action_space}')
print(f'Observation spasce: {env.observation_space}')

env.close()

Starting observation: {'h_edges': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], 'v_edges': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 'box_owner': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]}
action: [0 4 2]
Number of Claimed Edges: 60
[0 4 2]
0
action: [0 3 3]
Number of Claimed Edges: 59
[0 3 3]
1
action: [0 0 2]
Number of Claimed Edges: 58
[0 0 2]
0
action: [1 1 2]
Number of Claimed Edges: 57
[1 1 2]
action: [1 3 2]
Number of Claimed Edges: 56
[1 3 2]
action: [1 1 4]
Number of Claimed Edges: 55
[1 1 4]
action: [0 2 1]
Number of Claimed Edges: 54
[0 2 1]
0
action: [1 3 4]
Number of Claimed Edges: 53
[1 3 4]
action: [0 3 1]
Number of Claimed Edges: 52
[0 3 1]
0
action: [1 4 1]
Number of Claimed Edges: 51
[1 4 1]
action: [1 3 3]
Number of Claimed Edges: 50
[1 3 3]
action: [1 4 4]
Number of Claimed Edges: 49
[1 4 4]
action: [1 0 1]

: 